# 02 — Temporal EDA

## Objective
Characterise the temporal dynamics of the e-commerce stream:
- daily / hourly volume
- fraud rate evolution over time
- account-age distributions
- velocity patterns that will later become features

No modelling is performed; this notebook only informs feature design and split validity.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data_utils import load_processed
from temporal_features import add_temporal_features

df = load_processed("fraud_enriched")
df = add_temporal_features(df)
print(df.shape)
df.head(3)


## 1. Daily transaction volume & fraud rate

In [ ]:
daily = (df.set_index("purchase_time")
           .resample("D")
           .agg(n_tx=("user_id", "size"), fraud_rate=("class", "mean")))
daily["fraud_rate"] = daily["fraud_rate"].fillna(0)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(daily.index, daily["n_tx"], color="#4C78A8")
axes[0].set_ylabel("Transactions")
axes[0].set_title("Daily transaction volume")
axes[1].plot(daily.index, daily["fraud_rate"], color="#E45756")
axes[1].set_ylabel("Fraud rate")
axes[1].set_title("Daily fraud rate (label)")
plt.tight_layout()
plt.show()

print("Overall fraud rate:", df["class"].mean())
print("Peak daily volume:", daily["n_tx"].max())


## 2. Hour-of-day and day-of-week seasonality

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hour_rate = df.groupby("purchase_hour")["class"].mean()
hour_rate.plot(kind="bar", ax=axes[0], color="#4C78A8")
axes[0].set_title("Fraud rate by hour of day")
axes[0].set_xlabel("Hour (UTC)")

dow_rate = df.groupby("purchase_dayofweek")["class"].mean()
dow_rate.plot(kind="bar", ax=axes[1], color="#72B7B2")
axes[1].set_title("Fraud rate by day of week (0=Mon)")
plt.tight_layout()
plt.show()


## 3. Account age as a strong temporal signal

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for label, color in [(0, "#4C78A8"), (1, "#E45756")]:
    subset = df.loc[df["class"] == label, "account_age_hours"].clip(upper=200)
    subset.hist(bins=40, alpha=0.55, label=f"class={label}", color=color, ax=ax, density=True)
ax.set_xlabel("Account age (hours, clipped)")
ax.set_title("Account-age density by fraud label")
ax.legend()
plt.tight_layout()
plt.show()

print("Median account age (normal):", df.loc[df["class"]==0, "account_age_hours"].median())
print("Median account age (fraud): ", df.loc[df["class"]==1, "account_age_hours"].median())
print("Instant purchase rate (≤1s):", (df["account_age_seconds"] <= 1).mean())


## 4. Insights for feature engineering
- Very short account age is a powerful signal → `instant_purchase`, `account_age_hours`.
- Mild hour-of-day variation exists → keep `purchase_hour`, `is_night`.
- Fraud rate is relatively stable across the observation window → chronological split is reasonable.
- Downstream notebooks will compute velocity features (time since previous transaction per entity) inside each split.
